# 8. 텍스트 AI 기초 - 텍스트를 숫자로 바꾸는 법부터 GPT까지

> 이 노트북은 강의 자료에서 자동 변환되었습니다.  
> Google Colab에서 `런타임 > 런타임 유형 변경 > GPU`로 설정 후 실행하세요.


# 8. 텍스트 AI 기초 - 텍스트를 숫자로 바꾸는 법부터 GPT까지

## 학습 목표

1. 텍스트가 AI에게 어려운 이유(문맥 의존, 순서, 가변 길이)를 설명할 수 있다
2. 토큰화와 원-핫 인코딩의 개념과 한계를 이해할 수 있다
3. 임베딩이 원-핫의 한계를 어떻게 해결하는지 설명할 수 있다
4. RNN → LSTM → GRU의 발전 흐름과 각각의 해결 과제를 설명할 수 있다
5. Attention과 Transformer의 핵심 아이디어를 비유로 설명할 수 있다
6. BERT와 GPT의 차이를 구조와 학습 방식 측면에서 비교할 수 있다
7. GPT, ChatGPT, LLM의 차이를 구분할 수 있다

---

## 진행 순서

1. [왜 텍스트는 AI에게 어려운가?](#part1)
2. [Tokenization - "문장을 작은 조각으로 자른다"](#part2)
3. [원-핫 인코딩 - "이름표만 붙인다"](#part3)
4. [Embedding - "단어에 좌표를 붙인다"](#part4)
5. [RNN (Recurrent Neural Network) - "순서대로 읽는 AI"](#part5)
6. [LSTM (Long Short-Term Memory) - "기억 통로를 따로 만든다"](#part6)
7. [GRU (Gated Recurrent Unit) - "LSTM을 더 가볍게 줄인다"](#part7)
8. [Attention - "필요한 부분을 다시 본다"](#part8)
9. [Transformer - "전체 관계를 한 번에 본다"](#part9)
10. [BERT vs GPT - "독자 vs 작가"](#part10)
11. [GPT, ChatGPT, LLM은 같은 말이 아니다](#part11)
12. [오늘의 대화형 AI는 왜 더 똑똑해 보일까?](#part12)
13. [비판적으로 보기: 지금의 LLM도 한계가 있다](#part13)
14. [전체 흐름 요약](#part14)

---

## 1. 왜 텍스트는 AI에게 어려운가?

**학습목표**: 텍스트가 AI에게 어려운 3가지 이유를 설명할 수 있다

이미지는 픽셀, 소리는 파형처럼 비교적 바로 숫자로 다룰 수 있습니다.
하지만 텍스트는 단순히 숫자로 바꾸는 것만으로 끝나지 않습니다.

텍스트가 어려운 이유는 크게 3가지입니다.

| 문제 | 예시 | 왜 어려운가 |
|------|------|-------------|
| **문맥 의존** | "은행" = 금융기관 / 강가 | 같은 단어도 상황에 따라 뜻이 달라짐 |
| **순서 중요** | "개가 사람을 물었다" / "사람이 개를 물었다" | 단어가 같아도 순서가 바뀌면 의미가 달라짐 |
| **길이 가변** | 한 단어짜리 문장도 있고, 수백 단어 문장도 있음 | 입력 길이가 매번 다름 |

그래서 텍스트 AI는 보통 다음 순서로 발전했습니다.

1. 텍스트를 작은 조각으로 자른다
2. 그 조각을 숫자로 바꾼다
3. 조각들 사이의 관계를 학습한다
4. 문맥을 보고 이해하거나, 다음 내용을 생성한다

**핵심**: 텍스트는 문맥 의존, 순서 중요, 길이 가변이라는 3가지 어려움이 있어서, 단순히 숫자로 바꾸는 것 이상의 구조가 필요하다.

---

## 2. Tokenization - "문장을 작은 조각으로 자른다"

**학습목표**: 토큰화의 개념과 토큰 ≠ 단어인 이유를 이해할 수 있다

현대 텍스트 AI에서 가장 먼저 하는 일은 **토큰화(Tokenization)** 입니다.

즉, 문장을 모델이 다루기 쉬운 작은 단위인 **토큰(token)** 으로 쪼갭니다.

### 토큰은 꼭 "단어"와 같지 않다

예를 들어:

```
"인공지능을 공부합니다"
```

사람은 보통 이걸 한 문장으로 읽지만, 모델은 다음처럼 나눌 수 있습니다.

```
["인공지능", "을", "공부", "합니다"]
```

또는 더 잘게 나눌 수도 있습니다.

```
["인공", "지능", "을", "공부", "합니다"]
```

### 왜 굳이 잘게 나눌까?

| 이유 | 설명 |
|------|------|
| **희귀 단어 처리** | 처음 보는 긴 단어도 익숙한 조각으로 분해하면 다룰 수 있음 |
| **어휘 수 절약** | 모든 단어를 사전에 넣지 않아도 됨 |
| **현대 LLM 표준** | GPT(Generative Pre-trained Transformer), BERT(Bidirectional Encoder Representations from Transformers) 같은 모델은 보통 단어가 아니라 토큰 단위로 학습하고 생성함 |

### 꼭 기억할 점

- **토큰 = 단어**가 아닙니다.
- 모델의 입력 길이 제한도 보통 "**몇 단어**"가 아니라 "**몇 토큰**" 기준입니다.
- GPT류 모델은 보통 "**다음 단어**"가 아니라 "**다음 토큰**"을 예측합니다.

**핵심**: 토큰은 단어와 다르다. 현대 LLM은 토큰 단위로 학습하고 생성하며, 입력 길이 제한도 토큰 기준이다.

---

## 3. 원-핫 인코딩 - "이름표만 붙인다"

**학습목표**: 원-핫 인코딩의 원리와 한계를 설명할 수 있다

컴퓨터는 결국 숫자만 처리할 수 있습니다.
토큰을 숫자로 바꾸는 가장 단순한 방법은 **원-핫 인코딩(one-hot encoding)** 입니다.

```
사과   -> [1, 0, 0, 0, 0]
바나나 -> [0, 1, 0, 0, 0]
자동차 -> [0, 0, 1, 0, 0]
```

이 방식은 각 단어에 번호표를 주는 것과 비슷합니다.

### 한계

| 문제 | 설명 |
|------|------|
| **차원 폭발** | 단어가 10만 개면 벡터 길이도 10만 |
| **의미 부재** | 사과와 바나나는 둘 다 과일인데 벡터상 전혀 가깝지 않음 |
| **순서 무시** | 단어 자체만 표시할 뿐, 문장 흐름은 표현하지 못함 |

→ **"비슷한 단어는 비슷한 숫자로 표현할 수 없을까?"** → Embedding 등장

**핵심**: 원-핫 인코딩은 단순하지만, 차원 폭발과 의미 부재라는 한계가 있어 임베딩의 필요성을 만든다.

---

## 4. Embedding - "단어에 좌표를 붙인다"

**학습목표**: 임베딩이 원-핫의 한계를 해결하는 방식을 이해할 수 있다

> 분산 표현 아이디어는 더 일찍부터 있었고, 2013년 Word2Vec이 이를 널리 알렸습니다.
> 대표적 확산 계기: 2013년, *Efficient Estimation of Word Representations in Vector Space* (ICLR)
> 주요 연구자: Tomas Mikolov, Kai Chen, Greg S. Corrado, Jeffrey Dean
> 당시 소속: Google Inc., Mountain View, California
> 왜 중요한가: Google의 대규모 데이터와 계산 자원 위에서 "단어 의미를 벡터 공간으로 다룬다"는 생각을 NLP의 기본 문법으로 굳혔기 때문입니다.

### 핵심 아이디어

원-핫 같은 이름표를 **밀집 벡터(dense vector)** 로 바꿉니다.

```
원-핫:  사과 = [1, 0, 0, 0, 0]
임베딩: 사과 = [0.82, -0.15, 0.47]
```

### 비유

> 학생 이름표만 나눠주던 것에서,
> 이제는 교실 좌석 배치도를 만들어 비슷한 학생끼리 가까이 앉히는 것

### 좋은 점

- 비슷한 의미의 단어가 비슷한 위치에 놓일 수 있음
- 희소 벡터보다 훨씬 효율적임
- 오늘의 LLM(Large Language Model)도 결국은 `토큰 ID -> 임베딩 벡터` 단계로 시작함

### 유명한 예시

```
king - man + woman ≈ queen
```

이런 예시는 임베딩이 관계를 어느 정도 포착할 수 있음을 보여줍니다.
다만 이것이 "언어를 완전히 이해했다"는 뜻은 아닙니다.

### 남은 한계

| 한계 | 설명 |
|------|------|
| **문맥 무시** | "은행"이 금융인지 강가인지 구분 못 함 |
| **순서 무시** | 단어 좌표는 있어도 문장 흐름은 모름 |

→ **"순서를 따라 읽는 구조가 필요하다"** → RNN(Recurrent Neural Network) 등장

**핵심**: 임베딩은 단어를 밀집 벡터로 표현하여 의미적 유사성을 반영한다. 하지만 문맥과 순서는 여전히 다루지 못한다.

---

## 5. RNN (Recurrent Neural Network) - "순서대로 읽는 AI"

**학습목표**: RNN의 순차 처리 원리와 한계를 설명할 수 있다

> 대표 발표: 1990년, *Finding Structure in Time* (Cognitive Science)
> 주요 연구자: Jeffrey L. Elman
> 당시 소속: University of California, San Diego (UC San Diego)
> 왜 중요한가: 텍스트를 순서대로 읽으며 내부 상태를 갱신하는 신경망 관점을 널리 보여 주어, 이후 시퀀스 모델의 출발점이 되었기 때문입니다.

### 핵심 아이디어

RNN(Recurrent Neural Network)은 이전까지 읽은 정보를 **은닉 상태(hidden state)** 에 담아 다음 단어를 읽을 때 함께 참고합니다.

```
"나는" -> [은닉 상태 1]
"학교에" + [은닉 상태 1] -> [은닉 상태 2]
"간다" + [은닉 상태 2] -> [은닉 상태 3]
```

### 비유

> 한 줄짜리 메모장을 들고 책을 읽는 사람
> 새 내용을 적을수록 예전 메모는 점점 흐릿해진다

### 해결한 문제

- 문장을 순서대로 처리할 수 있음
- 길이가 다른 시퀀스도 다룰 수 있음
- "바로 앞에 무엇이 있었는가"를 어느 정도 반영할 수 있음

### 남은 한계

| 한계 | 설명 |
|------|------|
| **기울기 소실** | 긴 문장에서 앞부분 정보가 점점 사라짐 |
| **순차 처리** | 한 단어씩 읽어야 해서 느리고 병렬화가 어려움 |

### 기울기 소실 직관

전화 게임을 생각하면 쉽습니다.

- 5명쯤 지나면 원래 말과 비슷
- 50명쯤 지나면 많이 왜곡
- 100명쯤 지나면 거의 다른 말

RNN도 긴 거리를 거치며 정보를 전달할 때 비슷한 문제가 생깁니다.

→ **"긴 문장에서도 기억을 더 잘 남기자"** → LSTM(Long Short-Term Memory) 등장

**핵심**: RNN은 순서대로 읽으며 은닉 상태를 갱신하지만, 긴 문장에서 앞부분 정보가 점점 사라지는 기울기 소실 문제가 있다.

---

## 6. LSTM (Long Short-Term Memory) - "기억 통로를 따로 만든다"

**학습목표**: LSTM이 RNN의 기울기 소실 문제를 어떻게 완화하는지 이해할 수 있다

> 대표 발표: 1997년, *Long Short-Term Memory* (Neural Computation)
> 주요 연구자: Sepp Hochreiter, Jürgen Schmidhuber
> 당시 소속: Sepp Hochreiter - Technische Universität München, Jürgen Schmidhuber - IDSIA, Lugano, Switzerland
> 왜 중요한가: RNN의 긴 문장 기억 문제를 완화하는 핵심 장치를 제안해, 한동안 시퀀스 처리의 표준 해법으로 자리 잡았기 때문입니다.

### 핵심 아이디어

LSTM(Long Short-Term Memory)은 RNN에 **셀 상태(cell state)** 라는 장기 기억 통로와 여러 **게이트(gate)** 를 추가한 구조입니다.

| 게이트 | 비유 | 역할 |
|--------|------|------|
| **Forget Gate** | 지우개 | 옛 정보를 얼마나 지울지 결정 |
| **Input Gate** | 형광펜 | 새 정보를 얼마나 넣을지 결정 |
| **Output Gate** | 말하기 버튼 | 지금 무엇을 밖으로 내보낼지 결정 |

### 비유

> 메모를 그냥 하나만 쓰는 사람이 아니라,
> "버릴 것 / 남길 것 / 지금 말할 것"을 구분하는 숙련된 통역사

### 좋아진 점

- RNN보다 긴 의존관계를 더 잘 다룸
- 긴 문장에서 정보가 완전히 사라지는 문제를 완화함

### 남은 한계

| 한계 | 설명 |
|------|------|
| **구조가 복잡함** | 게이트가 많아 계산이 무거워짐 |
| **여전히 순차 처리** | 한 단어씩 읽는 구조는 그대로 |

→ **"조금 더 단순하게 줄일 수 없을까?"** → GRU(Gated Recurrent Unit) 등장

**핵심**: LSTM은 셀 상태와 게이트를 추가하여 긴 의존관계를 더 잘 다루지만, 구조가 복잡하고 여전히 순차 처리이다.

---

## 7. GRU (Gated Recurrent Unit) - "LSTM을 더 가볍게 줄인다"

**학습목표**: GRU가 LSTM을 어떻게 단순화했는지 설명할 수 있다

> 대표 발표: 2014년, *Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation* (EMNLP 2014)
> 주요 연구자: Kyunghyun Cho, Bart van Merriënboer, Caglar Gulcehre, Dzmitry Bahdanau, Fethi Bougares, Holger Schwenk, Yoshua Bengio
> 당시 소속: Cho / van Merriënboer / Gulcehre / Bengio - Université de Montréal, Bahdanau - Jacobs University Germany, Bougares / Schwenk - Université du Maine France
> 왜 중요한가: 몬트리올 신경망 번역 연구 흐름 속에서 LSTM의 핵심 아이디어를 더 단순하게 가져와, 실험과 응용에 쓰기 쉬운 대안을 제시했기 때문입니다.
> 참고: GRU는 이 논문에서 널리 알려졌고, 같은 시기 관련 후속 연구들에서 비교·정리되었습니다.

### 핵심 아이디어

GRU(Gated Recurrent Unit)는 LSTM(Long Short-Term Memory)을 단순화한 구조입니다.

- 게이트 수를 줄이고
- 셀 상태를 따로 두지 않고
- 더 단순한 방식으로 과거 정보와 새 정보를 섞습니다

| 항목 | LSTM | GRU |
|------|------|-----|
| 상태 | 셀 상태 + 은닉 상태 | 은닉 상태 중심 |
| 구조 | 더 복잡 | 더 단순 |
| 계산량 | 더 무거운 편 | 더 가벼운 편 |
| 성능 | 작업에 따라 다름 | 작업에 따라 다름 |

### 핵심 메시지

GRU는 "항상 더 좋다"도 아니고 "항상 더 나쁘다"도 아닙니다.
보통은 **더 단순하고 빠른 편**이라 실용적으로 자주 비교됩니다.

### 남은 한계

| 한계 | 설명 |
|------|------|
| **순차 처리** | 여전히 앞에서부터 차례대로 읽어야 함 |
| **먼 단어 연결의 어려움** | 앞 단어와 뒤 단어가 멀면 관계를 잡기 어려움 |

→ **"중간 단계를 다 거치지 말고, 중요한 부분을 직접 보자"** → Attention 등장

**핵심**: GRU는 LSTM을 단순화한 실용적 대안이다. 항상 더 좋거나 나쁜 것이 아니라, 작업에 따라 다르다.

---

## 8. Attention - "필요한 부분을 다시 본다"

**학습목표**: Attention이 등장한 배경과 핵심 아이디어를 이해할 수 있다

> 대표 발표: 2014년 arXiv 공개, 2015년 ICLR 발표, *Neural Machine Translation by Jointly Learning to Align and Translate*
> 주요 연구자: Dzmitry Bahdanau, Kyunghyun Cho, Yoshua Bengio
> 당시 소속: Bahdanau - Jacobs University Bremen, Germany / Cho, Bengio - Université de Montréal
> 왜 중요한가: "모든 정보를 한 벡터에 억지로 담지 말고, 필요한 부분을 다시 보자"는 발상을 정착시켜 Transformer로 가는 문을 열었기 때문입니다.

### 등장 배경: 문맥 벡터 병목

초기 Seq2Seq(sequence-to-sequence) 번역 모델은 입력 문장을 하나의 벡터로 압축한 뒤, 그 벡터만 보고 번역하려 했습니다.

```
입력 문장 전체 -> 문맥 벡터 1개 -> 출력 문장 생성
```

이 방식은 짧은 문장에서는 어느 정도 되지만, 긴 문장에서는 정보가 과하게 압축됩니다.

### 핵심 아이디어

출력 단어를 하나 만들 때마다,
입력 문장의 모든 단어를 다시 살펴보고
지금 가장 중요한 부분에 **집중(attend)** 합니다.

### 비유

> 메모 한 장만 보고 번역하던 통역사가,
> 이제는 원문 전체를 펼쳐 두고 필요한 문장을 다시 찾아보는 것

### 해결한 점

- 긴 문장에서 중요한 부분을 직접 참조할 수 있음
- 번역 성능이 크게 좋아짐
- "어디를 보고 있는가"를 비교적 해석하기 쉬움

### 남은 한계

| 한계 | 설명 |
|------|------|
| **RNN 기반 위에 얹힌 구조** | 뼈대는 여전히 순차적임 |
| **근본적 병렬화 한계** | 읽는 방식 자체는 여전히 느림 |

→ **"그렇다면 아예 RNN 없이 attention 중심으로 만들면?"** → Transformer 등장

**핵심**: Attention은 출력을 만들 때 입력 전체를 다시 살펴보는 발상으로, 긴 문장의 정보 압축 병목을 완화한다.

---

## 9. Transformer - "전체 관계를 한 번에 본다"

**학습목표**: Transformer의 Self-Attention과 Q-K-V 개념을 비유로 설명할 수 있다

> **대표 발표**: 2017년, *Attention Is All You Need* (당시 NIPS 2017, 현재 명칭 NeurIPS)
> **주요 연구자**: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin
> **당시 소속**: 논문에 직접 표시된 대표 소속은 Google Brain, Google Research, University of Toronto입니다.
> **왜 중요한가**: 현재 LLM의 기반 구조를 제시했고, 순차 처리의 병목을 줄이며 장거리 관계 학습과 학습 병렬화를 크게 개선했기 때문입니다.

Transformer의 핵심은 **Self-Attention(자기 주의 메커니즘)** 입니다.

즉, 문장 안의 각 토큰이 다른 토큰들과의 관련성을 계산해
문맥 속 자기 위치를 다시 정합니다.

### Self-Attention 직관

```
"나는 은행에서 돈을 찾았다"

은행 -> 돈(관련 큼), 찾았다(관련 큼)
     -> 그래서 "금융 기관" 쪽 뜻으로 해석
```

### Q-K-V를 검색엔진에 비유하면

| 요소 | 비유 | 역할 |
|------|------|------|
| **Q (Query)** | 내가 찾고 싶은 검색어 | 현재 토큰이 찾는 정보 |
| **K (Key)** | 각 페이지의 제목/태그 | 각 토큰이 "나는 이런 정보야"라고 내놓는 단서 |
| **V (Value)** | 실제 페이지 내용 | 실제로 가져올 정보 |

동작은 매우 단순하게 보면 이렇습니다.

1. 지금 토큰의 Q를 만든다
2. 다른 토큰들의 K와 비교한다
3. 점수가 높은 토큰의 V를 더 많이 가져온다

### RNN 계열과의 차이

| 항목 | RNN/LSTM/GRU | Transformer |
|------|--------------|-------------|
| 문장 처리 | 앞에서부터 순서대로 | 전체 관계를 함께 계산 |
| 먼 단어 연결 | 여러 단계를 거쳐야 함 | 직접 연결 가능 |
| 학습 병렬화 | 불리함 | 유리함 |
| 긴 의존관계 학습 | 어려운 편 | 상대적으로 유리 |

### 중요한 주의점

Transformer가 "**전체를 한 번에 본다**"는 말은 주로 **학습과 표현 계산** 관점에서의 장점입니다.

하지만 GPT처럼 텍스트를 실제로 생성할 때는:

```
이전 토큰들 -> 다음 토큰 1개 생성 -> 다시 다음 토큰 1개 생성
```

처럼 **자기회귀적으로 한 토큰씩** 이어서 만듭니다.

즉:

- 학습은 병렬화 이점이 큼
- 생성은 여전히 순차적임

이 차이를 구분해야 오해가 없습니다.

### 남은 한계

| 한계 | 설명 |
|------|------|
| **메모리/계산량 증가** | 기본 self-attention은 길이가 길수록 비용이 크게 늘어남 |
| **위치 정보 필요** | 단어 순서를 별도로 알려줘야 함 |
| **대규모 데이터와 계산 자원 필요** | 작은 데이터만으로는 강점이 잘 안 나옴 |

→ **"이 구조를 거대한 데이터로 미리 학습시키면?"** → BERT, GPT, 그리고 현대 LLM으로 이어짐

**핵심**: Transformer는 Self-Attention으로 전체 관계를 한 번에 계산한다. 학습은 병렬화 이점이 크지만, 생성은 여전히 순차적이다.

---

## 10. BERT (Bidirectional Encoder Representations from Transformers) vs GPT (Generative Pre-trained Transformer) - "독자 vs 작가"

**학습목표**: BERT와 GPT의 구조적 차이와 각각의 강점을 비교할 수 있다

### BERT (Bidirectional Encoder Representations from Transformers) - "빈칸 채우기로 문맥을 읽는다"

> 공개/발표: 2018년 10월 공개, 2019년 NAACL 논문 발표
> 주요 연구자: Jacob Devlin, Ming-Wei Chang, Kenton Lee, Kristina Toutanova
> 당시 소속: Google AI Language
> 왜 중요한가: 사전학습한 언어 표현을 다양한 이해 과제에 재사용하는 흐름을 폭발적으로 확산시켜, NLP 전이학습 시대를 본격화했기 때문입니다.

**학습 방식**: Masked Language Modeling (MLM, 마스크드 언어 모델링)

```
입력: "나는 오늘 [MASK]에서 커피를 마셨다"
정답: "카페"
```

BERT는 문장의 앞과 뒤를 함께 보며 빈칸을 맞힙니다.

비유하면:

> 시험지 전체를 보고 빈칸을 푸는 독자

### GPT (Generative Pre-trained Transformer) - "다음 토큰 예측으로 글쓰기를 배운다"

> 대표 발표: 2018년 6월, *Improving Language Understanding with Unsupervised Learning*
> 주요 연구자: Alec Radford, Karthik Narasimhan, Tim Salimans, Ilya Sutskever
> 당시 소속: OpenAI
> 왜 중요한가: 다음 토큰 예측을 대규모로 밀어붙이는 생성 중심 접근이 이후 GPT-2, GPT-3, ChatGPT 같은 흐름으로 이어졌기 때문입니다.

**학습 방식**: Causal Language Modeling, 즉 자기회귀(next-token prediction)

```
"오늘" -> "날씨가"
"오늘 날씨가" -> "매우"
"오늘 날씨가 매우" -> "화창합니다"
```

GPT는 왼쪽 문맥만 보고 다음 토큰을 예측합니다.

비유하면:

> 앞부분을 읽고 자연스럽게 다음 문장을 이어 쓰는 작가

### 비교

| 항목 | BERT | GPT |
|------|------|-----|
| 기본 구조 | Transformer 인코더 중심 | Transformer 디코더 중심 |
| 보는 방향 | 양방향 문맥 활용 | 왼쪽 문맥 기반 |
| 대표 학습 신호 | 빈칸 맞히기 | 다음 토큰 예측 |
| 태생적 강점 | 이해, 분류, 검색, 표현 추출 | 생성, 대화, 요약, 코딩 |

### 꼭 덧붙일 점

- BERT가 생성을 "절대 못 한다"는 뜻은 아닙니다.
- GPT가 이해를 "전혀 못 한다"는 뜻도 아닙니다.
- 다만 **태생적으로 더 잘 맞는 방향**이 다릅니다.

또한 실제 세상에는 BERT형, GPT형만 있는 것이 아니라
**인코더-디코더** 구조의 T5, BART 같은 계열도 있습니다.

**핵심**: BERT는 양방향 문맥으로 이해에 강하고, GPT는 왼쪽 문맥 기반으로 생성에 강하다. 태생적 강점이 다를 뿐 절대적 우열은 아니다.

---

## 11. GPT, ChatGPT, LLM은 같은 말이 아니다

**학습목표**: GPT, ChatGPT, LLM의 차이를 정확히 구분할 수 있다

비전공자가 가장 많이 헷갈리는 부분입니다.

### 1) GPT

GPT(Generative Pre-trained Transformer)는 원래 **모델 계열 이름**이자,
더 넓게는 "**다음 토큰을 예측하는 Transformer 기반 생성 모델 계열**"을 떠올리게 하는 이름입니다.

### 2) ChatGPT

ChatGPT는 **제품/서비스 이름**입니다.

즉, 단순히 "GPT라는 모델 하나"가 아니라:

- 대화형 인터페이스
- 지시를 따르도록 조정된 모델
- 안전장치
- 경우에 따라 도구 사용 기능

같은 요소가 합쳐진 **대화형 AI 서비스**입니다.

### 3) LLM

LLM(Large Language Model)은 대형 언어 모델의 총칭입니다.

여기에는 GPT 계열뿐 아니라 다른 회사의 다양한 모델도 들어갑니다.

### 한 줄 정리

| 용어 | 뜻 |
|------|----|
| **GPT (Generative Pre-trained Transformer)** | 모델 계열/아이디어 |
| **ChatGPT** | GPT 계열 모델을 활용한 대화형 제품 |
| **LLM (Large Language Model)** | 대형 언어 모델 전체를 가리키는 큰 범주 |

따라서:

- **ChatGPT = GPT 그 자체** 는 아님
- **LLM = GPT만 뜻하는 말** 도 아님

**핵심**: GPT는 모델 계열, ChatGPT는 대화형 서비스, LLM은 대형 언어 모델의 총칭이다. 세 용어를 혼동하지 않아야 한다.

---

## 12. 오늘의 대화형 AI는 왜 더 똑똑해 보일까?

**학습목표**: 현대 대화형 AI의 학습 과정(사전학습→지시학습→안전조정)을 이해할 수 있다

현대의 ChatGPT 같은 시스템은 보통 한 단계로 만들어지지 않습니다.

### 대략적인 흐름

1. **사전학습(Pretraining)**
   엄청난 양의 텍스트를 읽으며 다음 토큰 예측을 배웁니다.

2. **지시 따르기 학습(Instruction Tuning)**
   "요약해줘", "번역해줘", "표로 정리해줘" 같은 요청에 더 잘 반응하도록 조정합니다.

3. **대화 최적화 / 안전 조정**
   사람과 자연스럽게 대화하고, 위험한 답변을 줄이도록 다듬습니다.

즉, 오늘의 대화형 AI는
"거대한 언어 모델 + 사용하기 쉽게 조정된 대화 시스템"에 가깝습니다.

**핵심**: 현대 대화형 AI는 사전학습 → 지시 따르기 학습 → 안전 조정의 여러 단계를 거쳐 만들어진다.

---

## 13. 비판적으로 보기: 지금의 LLM도 한계가 있다

**학습목표**: 현재 LLM의 주요 한계(환각, 편향 등)를 인식할 수 있다

모델이 그럴듯하게 말한다고 해서, 인간처럼 완전히 이해한다고 단정하면 안 됩니다.

| 한계 | 설명 |
|------|------|
| **환각(Hallucination)** | 그럴듯하지만 틀린 내용을 지어낼 수 있음 |
| **컨텍스트 한계** | 한 번에 볼 수 있는 토큰 수에는 제한이 있음 |
| **편향과 데이터 한계** | 학습 데이터의 오류나 편향을 따라갈 수 있음 |
| **비용 문제** | 큰 모델일수록 메모리와 연산 비용이 큼 |
| **진짜 이해 논쟁** | 통계적 패턴 학습과 인간의 이해를 완전히 같다고 보기 어려움 |

그래서 LLM을 사용할 때는:

- 답이 자연스럽다고 바로 믿지 말고
- 출처를 확인하고
- 중요한 결정은 검증하는 습관이 필요합니다

**핵심**: LLM은 환각, 컨텍스트 한계, 편향, 비용 문제 등의 한계가 있으므로, 답을 바로 믿지 말고 검증하는 습관이 필요하다.

---

## 14. 전체 흐름 요약

**학습목표**: 텍스트 AI 발전의 전체 흐름을 시간순으로 정리할 수 있다

설명은 학습하기 쉬운 순서로 했지만, 실제 역사 흐름을 단순화하면 아래와 같습니다.

| 시기 | 핵심 아이디어 | 해결하려 한 문제 |
|------|---------------|------------------|
| 1990 | **RNN (Recurrent Neural Network)** | 순서를 따라 읽기 |
| 1997 | **LSTM (Long Short-Term Memory)** | 긴 문장에서 기억 유지 |
| 2000년대 초 | **분산 표현/임베딩 발전** | 단어 의미를 더 잘 표현 |
| 2013 | **Word2Vec** | 임베딩 대중화 |
| 2014 | **GRU (Gated Recurrent Unit)** | LSTM 단순화 |
| 2014 | **Attention** | 긴 문장 정보 압축 병목 완화 |
| 2017 | **Transformer** | 순차 처리 한계 돌파 |
| 2018 | **GPT (Generative Pre-trained Transformer)** | 다음 토큰 예측 기반 생성 사전학습 |
| 2018 공개 / 2019 발표 | **BERT (Bidirectional Encoder Representations from Transformers)** | 양방향 문맥 기반 이해 사전학습 |
| 2020년대 | **대화형 LLM(Large Language Model) 서비스** | 범용 생성, 질의응답, 도구 사용 |

### 누가 발표했나: 연도와 주요 연구자 한눈에 보기

아래 표의 "주요 연구자"는 **대표 논문 저자 기준**입니다.
실제 발전에는 더 많은 연구자와 후속 연구가 기여했습니다.

| 모델 | 대표 공개/발표 시점 | 주요 연구자 | 당시 소속(대표) |
|------|--------------------|-------------|------------------|
| **Embedding / Word2Vec** | 2013 | Tomas Mikolov, Kai Chen, Greg S. Corrado, Jeffrey Dean | Google Inc. |
| **RNN (Recurrent Neural Network)** | 1990 | Jeffrey L. Elman | UC San Diego |
| **LSTM (Long Short-Term Memory)** | 1997 | Sepp Hochreiter, Jürgen Schmidhuber | Technische Universität München, IDSIA |
| **GRU (Gated Recurrent Unit)** | 2014 | Kyunghyun Cho, Bart van Merriënboer, Caglar Gulcehre, Dzmitry Bahdanau, Fethi Bougares, Holger Schwenk, Yoshua Bengio | Université de Montréal, Jacobs University, Université du Maine |
| **Attention** | 2014 arXiv / 2015 ICLR | Dzmitry Bahdanau, Kyunghyun Cho, Yoshua Bengio | Jacobs University Bremen, Université de Montréal |
| **Transformer** | 2017 | Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin | Google Brain, Google Research, University of Toronto |
| **GPT (Generative Pre-trained Transformer)** | 2018 | Alec Radford, Karthik Narasimhan, Tim Salimans, Ilya Sutskever | OpenAI |
| **BERT (Bidirectional Encoder Representations from Transformers)** | 2018 공개 / 2019 논문 | Jacob Devlin, Ming-Wei Chang, Kenton Lee, Kristina Toutanova | Google AI Language |

### 외울 때는 이렇게 기억하기

- **Embedding / Word2Vec**: "단어 의미를 숫자 좌표로 바꾸는 시대를 열었다."
- **RNN**: "텍스트를 순서대로 읽는 신경망의 출발점이다."
- **LSTM**: "긴 문장을 더 오래 기억하게 만든 개선형 RNN이다."
- **GRU**: "LSTM을 더 단순하고 가볍게 만든 실용형 대안이다."
- **Attention**: "필요한 입력을 다시 보는 발상으로 병목을 깨기 시작했다."
- **Transformer**: "RNN 없이 attention 중심으로 현대 LLM의 뼈대를 세웠다."
- **BERT**: "사전학습한 언어 이해 모델을 여러 과제에 재사용하는 흐름을 키웠다."
- **GPT**: "다음 토큰 예측 기반 생성 모델이 얼마나 강력한지 보여 주었다."

### 모델들이 자기소개를 한다면

**RNN**: "나는 순서를 따라 읽을 수 있어. 하지만 긴 문장 앞부분은 자주 잊어."

**LSTM**: "나는 기억 통로를 따로 둬서 오래된 정보도 좀 더 버틸 수 있어."

**GRU**: "나는 LSTM보다 단순해서 가볍게 써보기 좋아."

**Attention**: "지금 필요한 부분을 다시 찾아보자. 다 기억하려고만 하지 말고."

**Transformer**: "관계를 전체적으로 보자. 멀리 떨어진 단어도 직접 연결할 수 있어."

**GPT**: "나는 다음 토큰을 예측하며 글쓰기를 배웠어."

**ChatGPT 같은 서비스**: "나는 거대한 언어 모델을 대화형으로 쓰기 쉽게 다듬은 결과물이야."

---

#### 참고

- Bengio et al. (2003), *A Neural Probabilistic Language Model*
  https://www.jmlr.org/papers/v3/bengio03a.html
- Mikolov et al. (2013), *Efficient Estimation of Word Representations in Vector Space*
  https://research.google/pubs/efficient-estimation-of-word-representations-in-vector-space/
- Elman (1990), *Finding Structure in Time*
- Hochreiter & Schmidhuber (1997), *Long Short-Term Memory*
- Cho et al. (2014), *Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation*
  https://aclanthology.org/D14-1179/
- Bahdanau et al. (2014), *Neural Machine Translation by Jointly Learning to Align and Translate*
  https://arxiv.org/abs/1409.0473
- Vaswani et al. (2017), *Attention Is All You Need*
  https://arxiv.org/abs/1706.03762
- Devlin et al. (2019), *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*
  https://aclanthology.org/N19-1423/
- Radford et al. (2018), *Improving Language Understanding with Unsupervised Learning*
  https://openai.com/index/language-unsupervised/
- Hugging Face Transformers, *Tokenization summary*
  https://huggingface.co/docs/transformers/main/en/tokenizer_summary
- Hugging Face Transformers, *Causal language modeling*
  https://huggingface.co/docs/transformers/en/tasks/language_modeling
- Hugging Face Transformers, *Masked language modeling*
  https://huggingface.co/docs/transformers/en/tasks/masked_language_modeling
- OpenAI Developers, *Text generation guide*
  https://developers.openai.com/api/docs/guides/text
